[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 困难: 字节对编码 (BPE)

实现一个简单的**BPE 分词器**——GPT/LLaMA 分词化的基础。

### 函数签名
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### 算法 (训练)
1. 将每个单词拆分为字符 + `</w>` 结束标记
2. 统计语料库中所有相邻字符对
3. 将最频繁的字符对合并为单个 token
4. 重复 `num_merges` 次

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
# 无需导入

In [ ]:
# ✏️ 在此实现你的代码

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        pass  # 迭代查找并合并最频繁的字符对

    def encode(self, text):
        pass  # 应用已学习的合并来分割文本

## WordPiece vs BPE：详细对比

让我详细讲解一下WordPiece和BPE（Byte Pair Encoding）这两种主流分词算法的区别。

### 1. 基本概念

### BPE (Byte Pair Encoding)
- **提出者**：最初用于数据压缩，后被引入NLP（2016年，Sennrich et al.）
- **核心思想**：基于频率统计，迭代合并最频繁的字符对
- **代表模型**：GPT系列、LLaMA、RoBERTa

#### WordPiece
- **提出者**：Google（2016年，Schuster & Nakajima）
- **核心思想**：基于似然度提升，选择合并后能最大程度增加语言模型似然度的子词对
- **代表模型**：BERT、ALBERT、DistilBERT

### 2. 核心算法差异

#### BPE的合并策略
```python
# BPE：选择频率最高的 pair
pair_freq = count_pairs(corpus)
most_frequent = max(pair_freq, key=lambda x: x[1])
# 直接合并最频繁的 pair
merge(most_frequent)
```

#### WordPiece的合并策略
```python
# WordPiece：选择能最大化似然度的 pair
for each pair in all_pairs:
    # 计算合并后的似然度提升
    likelihood_gain = calculate_likelihood(corpus, pair)
# 选择增益最大的 pair
best_pair = max(pairs, key=lambda x: likelihood_gain[x])
merge(best_pair)
```

### 3. 训练过程对比

#### BPE训练流程
```
1. 初始化：所有词拆分为字符 + </w>
2. 统计所有相邻字符对的频率
3. 选择频率最高的字符对进行合并
4. 更新词汇表
5. 重复步骤2-4 N次
```

#### WordPiece训练流程
```
1. 初始化：所有词拆分为字符
2. 构建初始词汇表（所有字符）
3. 对每个候选合并：
   a. 尝试合并 pair
   b. 计算训练数据的似然度
   c. 计算似然度增益
4. 选择增益最大的pair进行合并
5. 更新词汇表
6. 重复步骤3-5 N次
```

### 4. 代码示例对比

#### BPE实现（简化版）
```python
class BPE:
    def train(self, corpus, num_merges):
        # 初始化：字符级拆分
        word_freqs = self._init_word_freqs(corpus)
        
        for _ in range(num_merges):
            # 统计pair频率
            pair_freqs = defaultdict(int)
            for word, freq in word_freqs.items():
                for i in range(len(word)-1):
                    pair_freqs[(word[i], word[i+1])] += freq
            
            # 选择最频繁的pair
            best_pair = max(pair_freqs, key=lambda x: pair_freqs[x])
            
            # 合并
            self.merges[best_pair] = ''.join(best_pair)
            word_freqs = self._apply_merge(word_freqs, best_pair)
```

#### WordPiece实现（简化版）
```python
class WordPiece:
    def train(self, corpus, num_merges):
        # 初始化：字符级拆分
        self.vocab = set(all_characters)
        word_freqs = self._init_word_freqs(corpus)
        
        for _ in range(num_merges):
            best_gain = -float('inf')
            best_pair = None
            
            # 对所有可能的pair评估
            for pair in self._get_candidate_pairs(word_freqs):
                # 计算似然度增益
                gain = self._compute_likelihood_gain(word_freqs, pair)
                if gain > best_gain:
                    best_gain = gain
                    best_pair = pair
            
            # 合并增益最大的pair
            self.merges[best_pair] = ''.join(best_pair)
            self.vocab.add(''.join(best_pair))
            word_freqs = self._apply_merge(word_freqs, best_pair)
    
    def _compute_likelihood_gain(self, word_freqs, pair):
        """计算合并pair后似然度的提升"""
        # 计算当前似然度
        current_likelihood = self._calculate_likelihood(word_freqs)
        
        # 模拟合并后的似然度
        new_word_freqs = self._apply_merge(word_freqs, pair)
        new_likelihood = self._calculate_likelihood(new_word_freqs)
        
        return new_likelihood - current_likelihood
```

### 5. 关键区别总结

| 维度 | BPE | WordPiece |
|------|-----|-----------|
| **选择标准** | 频率最高 | 似然度增益最大 |
| **计算复杂度** | O(n) - 较低 | O(n²) - 较高 |
| **训练速度** | 快 | 慢（需要计算似然度） |
| **词汇表质量** | 好 | 更好（理论上最优） |
| **实现难度** | 简单 | 复杂 |
| **中文支持** | 需要字节级处理 | 需要字节级处理 |

### 6. 实际应用中的差异

#### 分词效果对比
```python
# 示例文本："lower"
# BPE (假设训练在英语语料上)
"lower" ==> ["lo", "wer"]  # 基于频率

# WordPiece (假设训练在英语语料上)  
"lower" ==> ["low", "er"]  # 基于似然度，能更好理解词根

# 示例文本："unhappiness"
# BPE可能：
["un", "ha", "pp", "iness"]  # 纯粹基于频率

# WordPiece可能：
["un", "happi", "ness"]  # 基于语言模型，更好捕捉语义
```

#### 优缺点对比

**BPE优点**：
- 训练速度快
- 实现简单
- 内存占用小
- 适合大规模训练

**BPE缺点**：
- 可能产生次优的切分
- 对罕见词处理不够好
- 频率不等于语言学合理性

**WordPiece优点**：
- 更符合语言学直觉
- 能更好地处理形态学变化
- 词汇表更高效

**WordPiece缺点**：
- 训练计算量大
- 实现复杂
- 需要额外的似然度计算

### 7. 现代发展趋势

1. **SentencePiece**：结合了BPE和WordPiece的优点，支持无空格语言（如中文、日文）

2. **Unigram**：另一种基于概率的分词方法，使用EM算法

3. **BBPE (Byte-level BPE)**：
   - GPT-2/3/4使用
   - 在字节级别操作
   - 支持所有语言和emoji
   - 无需预分词

```python
# Byte-level BPE示例
# 文本 -> UTF-8字节 -> BPE合并
"你好" ==> [b'\xe4', b'\xbd', b'\xa0'] ==> ["ä½", " "] 
# (实际更复杂)
```

### 8. 选择建议

**选择BPE的场景**：
- 大规模训练
- 资源有限
- 主要处理英语等空格分隔语言
- 对训练时间敏感

**选择WordPiece的场景**：
- 需要更高质量的分词
- 有充足的计算资源
- 处理形态丰富的语言
- 对模型性能要求高

**选择SentencePiece的场景**：
- 多语言模型
- 中文、日文等无空格语言
- 希望统一处理方式

总的来说，两种方法各有优势，现代实践更倾向于使用Byte-level BPE（如GPT系列）或SentencePiece（如T5、LLaMA），因为它们更通用且效果更好。

In [ ]:
from collections import defaultdict
import re

class SimpleBPE:
    def __init__(self):
        self.merges = {}  # 存储合并规则: (pair) -> new_token
        self.vocab = set()  # 词汇表
        self.word_splits = {}  # 缓存单词的拆分结果
        
    def train(self, corpus: list[str], num_merges: int):
        """
        训练BPE分词器
        
        Args:
            corpus: 训练语料库
            num_merges: 合并次数
        """
        # 初始化：将每个单词拆分为字符 + </w>
        word_freqs = defaultdict(int) # 单词的频率
        for text in corpus:
            # 分词（按空格分割）
            for word in text.split():
                # 将单词拆分为字符并添加结束标记
                chars = list(word) + ['</w>']
                word_freqs[tuple(chars)] += 1
        
        # 执行合并操作
        for merge_idx in range(num_merges):
            # 统计所有相邻字符对的频率
            pair_freqs = defaultdict(int)
            
            for word, freq in word_freqs.items():
                # 遍历单词中的相邻字符对
                for i in range(len(word) - 1):
                    pair = (word[i], word[i+1])
                    pair_freqs[pair] += freq
            
            if not pair_freqs:
                break
                
            # 找到最频繁的字符对
            most_frequent_pair = max(pair_freqs.items(), key=lambda x: x[1])[0]
            
            # 记录合并规则
            new_token = ''.join(most_frequent_pair)
            self.merges[most_frequent_pair] = new_token
            self.vocab.add(new_token)
            
            # 更新所有单词：合并最频繁的字符对
            new_word_freqs = defaultdict(int)
            for word, freq in word_freqs.items():
                new_word = []
                i = 0
                while i < len(word):
                    # 检查当前字符对是否匹配
                    if i < len(word) - 1 and (word[i], word[i+1]) == most_frequent_pair:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_word_freqs[tuple(new_word)] += freq
            
            word_freqs = new_word_freqs
        
        # 将单个字符也加入词汇表
        for word in word_freqs.keys():
            for token in word:
                if token != '</w>':
                    self.vocab.add(token)
        
        # 缓存常用的单词拆分结果
        self._build_cache()
    
    def _build_cache(self):
        """构建单词拆分的缓存"""
        # 这里可以预计算一些常用单词的拆分
        pass
    
    def _split_word(self, word: str) -> list[str]:
        """
        使用训练好的合并规则拆分单个单词
        """
        # 初始化：将单词拆分为字符 + </w>
        tokens = list(word) + ['</w>']
        
        # 应用合并规则（按训练顺序应用）
        # 注意：需要按照训练时的顺序应用合并
        for pair, merged in self.merges.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        
        # 移除结束标记
        if tokens and tokens[-1] == '</w>':
            tokens = tokens[:-1]
        
        return tokens
    
    def encode(self, text: str) -> list[str]:
        """
        将文本编码为token序列
        
        Args:
            text: 输入文本
            
        Returns:
            token列表
        """
        if not text:
            return []
        
        # 按空格分割文本
        words = text.split()
        result = []
        
        for word in words:
            # 拆分每个单词
            tokens = self._split_word(word)
            result.extend(tokens)
        
        return result


In [ ]:
# 🧪 调试
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('合并:', bpe.merges[:5])
print('编码:', bpe.encode('low lower'))

In [ ]:
# ✅ 提交
from torch_judge import check
check('bpe')